# Lab 09-11 · CMOS and Scaling

Companion to chapter [09-11](../../chapters/09-11-cmos-and-scaling.md). Handout: [README.md](README.md).

In [1]:
import math
from pathlib import Path
import numpy as np
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VDD, VTN, VTP = 1.0, 0.3, 0.3

def i_sq(vgs, vds, k, vt):
    # square-law drain current (m = 1), zero below threshold
    vov = vgs - vt
    if vov <= 0:
        return 0.0
    return k * (vov * vds - vds ** 2 / 2) if vds < vov else k * vov ** 2 / 2

def vout(vin, r):
    # output voltage where the NMOS (k_n = r) sinks what the PMOS (k_p = 1) sources
    if vin <= VTN:
        return VDD
    if vin >= VDD - VTP:
        return 0.0
    return brentq(lambda vo: r * i_sq(vin, vo, 1, VTN) - i_sq(VDD - vin, VDD - vo, 1, VTP), 0, VDD, xtol=1e-14)

## Part 1 — Transfer curves and noise margins

$r = k_n/k_p$. Equal widths give $r = \mu_n/\mu_p$ = 2; $W_p = 2W_n$ gives $r$ = 1.

*Predict first:* does doubling the PMOS width move $V_M$ by more or less than 50 mV?

In [2]:
def vm(r):
    return brentq(lambda v: vout(v, r) - v, VTN + 1e-9, VDD - VTP - 1e-9, xtol=1e-14)
vm2, vm1 = vm(2.0), vm(1.0)
print(f"V_M: equal widths {vm2:.4f} V, W_p = 2 W_n {vm1:.4f} V, shift {1e3*(vm1-vm2):.1f} mV")
assert abs(vm2 - 0.4657) < 5e-5 and abs(vm1 - 0.500) < 5e-4

vin = np.linspace(0, 1, 20001)
vo = np.array([vout(x, 1.0) for x in vin])
g = np.gradient(vo, vin)
idx = np.where(g < -1)[0]
vil, vih = vin[idx[0]], vin[idx[-1]]
print(f"V_IL = {vil:.3f} V, V_IH = {vih:.3f} V, NM_L = {vil:.3f} V, NM_H = {VDD - vih:.3f} V")
assert abs(vil - 0.450) < 5e-4 and abs(vih - 0.550) < 5e-4

for r in (0.5, 1.0, 2.0, 4.0):
    plt.plot(vin, [vout(x, r) for x in vin], label=f"k_n/k_p = {r}")
plt.plot([0, 1], [0, 1], "k:", lw=0.8)
plt.xlabel("V_in (V)"); plt.ylabel("V_out (V)"); plt.legend(); plt.show()

V_M: equal widths 0.4657 V, W_p = 2 W_n 0.5000 V, shift 34.3 mV
V_IL = 0.450 V, V_IH = 0.550 V, NM_L = 0.450 V, NM_H = 0.450 V


### Short-channel sizing

With the velocity-saturated model of 09-10 ($L$ = 50 nm, 2 nm EOT, $m$ = 1.2), the PMOS penalty is smaller than the mobility ratio (Problem 09-11.12).

In [3]:
COX = 3.9 * 8.8541878128e-14 / 2e-7
def isat_vs(vov, mu, vs, L=50e-7, m=1.2, W=1e-4):
    es = 2 * vs / mu
    vd = 1 / (m / vov + 1 / (es * L))
    return W * COX * vs * (vov - m * vd)
ratio_on = isat_vs(0.55, 183, 8e6) / isat_vs(0.55, 61, 6e6)
ratio_vm = isat_vs(0.15, 183, 8e6) / isat_vs(0.15, 61, 6e6)
print(f"W_p/W_n for equal delays {ratio_on:.2f}; for V_M = V_DD/2 {ratio_vm:.2f}; mobility ratio 3")
assert abs(ratio_on - 2.15) < 5e-3 and abs(ratio_vm - 2.63) < 5e-3

W_p/W_n for equal delays 2.15; for V_M = V_DD/2 2.63; mobility ratio 3


## Part 2 — A ring oscillator, transition by transition

Each stage switches its output with a constant current once its input crosses $V_{DD}/2$; the output reaches $V_{DD}/2$ after $CV_{DD}/2I$. Follow one transition around the ring.

*Predict first:* is the period of a 101-stage ring 101 or 202 stage delays?

In [4]:
def ring_period(N, C, V, In, Ip, turns=6):
    t, falling, crossings = 0.0, True, []
    # node 0 starts a falling transition at t = 0; each stage inverts the direction
    for k in range(N * turns):
        t += C * V / (2 * (In if falling else Ip))
        falling = not falling
        if (k + 1) % N == 0:          # back at node 0
            crossings.append(t)
    # node 0 completes a full cycle every two passes
    return crossings[2] - crossings[0]
T = ring_period(101, 1e-15, 0.75, 100e-6, 80e-6)
taud = 1e-15 * 0.75 / 4 * (1 / 100e-6 + 1 / 80e-6)
print(f"period {T*1e12:.1f} ps = {T/taud:.1f} stage delays; frequency {1/T/1e9:.3f} GHz")
assert abs(T * 1e12 - 852) < 0.5 and abs(T / taud - 202) < 1e-9

period 852.2 ps = 202.0 stage delays; frequency 1.173 GHz


## Part 3 — Scaling calculator

Dimensions scale by $1/\kappa$ per node, voltages by $1/u$; frequency rises as $1/$delay ($\propto \kappa$). Power density grows as $\kappa^2/u^2$ per node.

*Predict first:* with the supply held at 1.2 V over five 0.7× nodes, by what factor does power density grow?

In [5]:
k = 1 / 0.7
nodes = np.arange(0, 6)
for u, lab in ((k, "constant field"), (1 / 0.9, "V x 0.9 per node"), (1.0, "constant V")):
    pd = (k ** 2 / u ** 2) ** nodes
    plt.semilogy(nodes, pd, marker="o", label=lab)
    print(f"{lab:18s}: power density after 5 nodes x{pd[-1]:.1f}")
growth = (k ** 2) ** 5
assert abs(growth - 35.4) < 0.05
plt.xlabel("nodes"); plt.ylabel("relative power density"); plt.legend(); plt.show()

constant field    : power density after 5 nodes x1.0
V x 0.9 per node  : power density after 5 nodes x12.3
constant V        : power density after 5 nodes x35.4


## Part 4 — Fifty years of processors

K. Rupp's dataset (CC-BY 4.0), in `data/`.

*Predict first:* in which five-year window does the median clock stop rising?

In [6]:
f = np.loadtxt(Path("data") / "rupp_frequency_mhz.dat", comments="#")
w = np.loadtxt(Path("data") / "rupp_power_w.dat", comments="#")
def med(a, lo, hi):
    return float(np.median(a[(a[:, 0] >= lo) & (a[:, 0] < hi), 1]))
for lo in (1995, 2000, 2005, 2010, 2015):
    print(f"{lo}-{lo+4}: median clock {med(f, lo, lo+5):7.0f} MHz, median power {med(w, lo, lo+5):6.1f} W")
assert abs(med(f, 2000, 2005) - 1811) < 0.5 and abs(med(f, 2005, 2010) - 2642) < 0.5
assert abs(med(f, 2010, 2015) - 2800) < 0.5 and abs(f[:, 1].max() - 4656) < 0.5
assert abs(med(w, 2000, 2005) - 74.0) < 0.05 and abs(med(w, 2015, 2022) - 205) < 0.5
fig, ax = plt.subplots()
ax.semilogy(f[:, 0], f[:, 1], "o", ms=3, label="clock (MHz)")
ax.semilogy(w[:, 0], w[:, 1], "s", ms=3, label="typical power (W)")
ax.axvline(2004, color="k", ls=":")
ax.set_xlabel("year"); ax.legend(); plt.show()

1995-1999: median clock     297 MHz, median power   27.4 W
2000-2004: median clock    1811 MHz, median power   74.0 W
2005-2009: median clock    2642 MHz, median power   95.6 W
2010-2014: median clock    2800 MHz, median power  135.0 W
2015-2019: median clock    2225 MHz, median power  205.0 W
